In [0]:
from pyspark.sql import functions as F

CATALOG = "Workspace"

SCHEMA = "retail_sales"

BASE_SCHEMA = f"{CATALOG}.{SCHEMA}"

PRACTICE_TABLE = (
    f"{BASE_SCHEMA}.delta_sales_practice"
)

print("Delta Lake 5")
print("Practice table:", PRACTICE_TABLE)

In [0]:
silver_df = spark.table(
    f"{BASE_SCHEMA}.silver_store_sales"
)

print("Silver sales rows:", silver_df.count())

display(silver_df.limit(5))

In [0]:
table_details = spark.sql(
    f"DESCRIBE DETAIL {BASE_SCHEMA}.silver_store_sales"
)

display(table_details)

In [0]:
history_df = spark.sql(
    f"DESCRIBE HISTORY {BASE_SCHEMA}.silver_store_sales"
)
display(history_df)

In [0]:
practice_data = [
    (1001, "Laptop", 2, 800.00),
    (1002, "Keyboard", 5, 40.00),
    (1003, "Mouse", 10, 20.00),
    (1004, "Monitor", 3, 250.00),
    (1005, "Headphones", 4, 60.00)
]

columns = [
    "transaction_id",
    "product_name",
    "quantity",
    "unit_price"
]

practice_df = spark.createDataFrame(
    practice_data,
    columns
)

practice_df = practice_df.withColumn(
    "total_amount",
    F.round(
        F.col("quantity") * F.col("unit_price"),
        2
    )
)

display(practice_df)

In [0]:
(
    practice_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(PRACTICE_TABLE)
)

print("Practice Delta table created!")

display(
    spark.table(PRACTICE_TABLE)
)

In [0]:
display(
    spark.sql(
        f"DESCRIBE HISTORY {PRACTICE_TABLE}"
    )
)

In [0]:
spark.sql(f"""
UPDATE {PRACTICE_TABLE}
SET
    quantity = 8,
    total_amount = 8 * unit_price
    WHERE
    transaction_id = 1002          
          """)

display(
    spark.table(PRACTICE_TABLE)
    .filter("transaction_id = 1002")
)

In [0]:
display(
    spark.sql(
        f"DESCRIBE HISTORY {PRACTICE_TABLE}"
    )
)

In [0]:
spark.sql(f"""
        DELETE FROM {PRACTICE_TABLE}
        WHERE transaction_id = 1005
        """)
display(
    spark.table(PRACTICE_TABLE)
    .orderBy("transaction_id")
)

In [0]:
history = spark.sql(
    f"DESCRIBE HISTORY {PRACTICE_TABLE}"
)

display(
    history.select(
        "version",
        "timestamp",
        "operation"
    )
)

current_version = history.agg(
    F.max("version")
).first()[0]

print("Current version:", current_version)

In [0]:
previous_version = current_version - 1
print("Previous version:", previous_version)

In [0]:
previous_df = spark.sql(f"""
SELECT *
FROM {PRACTICE_TABLE}
VERSION AS OF {previous_version}
ORDER BY transaction_id
 """)

display(previous_df)

In [0]:
current_count = spark.table(
    PRACTICE_TABLE
).count()

previous_count = previous_df.count()

print("Current rows:", current_count)
print("Previous version rows:", previous_count)

In [0]:
new_sales_data = [
    (1002, "Keyboard", 10, 40.00),
    (1006, "Webcam", 2, 120.00),
    (1007, "USB Hub", 3, 35.00)
]

new_sales_df = spark.createDataFrame(
    new_sales_data,
    columns
).withColumn(
    "total_amount",
    F.round(
        F.col("quantity") * F.col("unit_price"),
        2
    )
)

display(new_sales_df)

In [0]:
new_sales_df.createOrReplaceTempView(
    "incoming_sales"
)

print("Incoming sales view created!")

In [0]:
spark.sql(f"""
MERGE INTO {PRACTICE_TABLE} AS target
USING incoming_sales AS source

ON target.transaction_id = source.transaction_id

WHEN MATCHED THEN
    UPDATE SET
        target.product_name = source.product_name,
        target.quantity = source.quantity,
        target.unit_price = source.unit_price,
        target.total_amount = source.total_amount

WHEN NOT MATCHED THEN
    INSERT (
        transaction_id,
        product_name,
        quantity,
        unit_price,
        total_amount
    )
    VALUES(
        source.transaction_id,
        source.product_name,
        source.quantity,
        source.unit_price,
        source.total_amount
    )
  """)
print("MERGE completed successfully!")

In [0]:
display(
    spark.table(PRACTICE_TABLE)
    .orderBy("transaction_id")
)

In [0]:
final_df = spark.table(PRACTICE_TABLE)

print("Total transactions:", final_df.count())

print(
    "Unique transactions:",
    final_df.select("transaction_id").distinct().count()
)

In [0]:
display(
    final_df.filter(
        F.col("transaction_id") == 1002
    )
)

In [0]:
final_df = spark.table(PRACTICE_TABLE)

total_rows = final_df.count()

distinct_ids = (
    final_df.select("transaction_id")
    .distinct()
    .count()
)

invalid_rows = final_df.filter(
    (F.col("quantity") <= 0) |
    (F.col("unit_price") < 0) |
    F.col("transaction_id").isNull()
).count()

keyboard = (
    final_df
    .filter(F.col("transaction_id") == 1002)
    .first()
)

assert total_rows == 6
assert total_rows == distinct_ids
assert invalid_rows == 0
assert keyboard is not None
assert keyboard["quantity"] == 10
assert keyboard["total_amount"] == 400.0

print("All Delta Lake validation checks passed!")

In [0]:
count_before = spark.table(PRACTICE_TABLE).count()

spark.sql(f"""
MERGE INTO {PRACTICE_TABLE} AS target
USING incoming_sales AS source

ON target.transaction_id = source.transaction_id

WHEN MATCHED THEN UPDATE SET
    target.product_name = source.product_name,
    target.quantity = source.quantity,
    target.unit_price = source.unit_price,
    target.total_amount = source.total_amount


WHEN NOT MATCHED THEN INSERT (
    transaction_id,
    product_name,
    quantity,
    unit_price,
    total_amount
)

VALUES (
    source.transaction_id,
    source.product_name,
    source.quantity,
    source.unit_price,
    source.total_amount
)
""")

count_after = spark.table(PRACTICE_TABLE).count()

print("Before rerun:", count_before)
print("After rerun:", count_after)

assert count_before == count_after == 6

print("Idempotent row - count check passed!")

In [0]:
spark.sql(f"""
RESTORE TABLE {PRACTICE_TABLE}
TO VERSION AS OF 3
""")

display(
    spark.table(PRACTICE_TABLE)
    .orderBy("transaction_id")
)

In [0]:
display(
    spark.sql(
        f"DESCRIBE HISTORY {PRACTICE_TABLE}"
    ).select(
        "version",
        "timestamp",
        "operation"
    )
)